<a href="https://colab.research.google.com/github/JSJeong-me/MCP/blob/main/02%20Agent%20Tool/7_production_hardening_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 7 실습: 보안 및 운영 안정화 (Production Hardening)

이 노트북은 **Module 7: 보안 및 운영 안정화** 실습용입니다.

학습 목표:
- **사용자 동의(Consent)** 기반 실행의 의미 이해
- **최소 권한(Least Privilege)** 접근 제어 이해
- **OAuth 2.1 스타일 토큰 검증**의 핵심 요소(만료, audience, scope) 이해
- **에러 처리, 감사 로그(Audit Logging), 속도 제한(Rate Limiting), 성능 측정** 패턴 이해
- Colab에서 바로 실행 가능한 **hardened MCP demo server** 실습


## 실행 순서

아래 순서대로 실행하세요.

1. **FastMCP 설치**
2. **기본 import**
3. **샘플 SQLite DB 생성**
4. **보안/운영 유틸리티 정의**
5. **MCP Server 정의**
6. **Client 생성**
7. **정상 read 호출 테스트**
8. **동의 없이 export 호출 → 실패 확인**
9. **동의 부여 후 export 호출 → 성공 확인**
10. **만료 토큰 / scope 부족 / audience 오류 테스트**
11. **rate limit 초과 테스트**
12. **audit log 조회**
13. **실습 정리**

권장:
- 메뉴에서 **런타임 → 모두 실행**
- 또는 위에서 아래로 한 셀씩 실행


## 설계 포인트

이번 노트북은 Colab에서 돌아가도록 **실제 브라우저 OAuth 로그인 대신 mock access token**을 사용합니다.

하지만 다음 production hardening 패턴은 실제와 같은 흐름으로 구현합니다.

- 토큰 검증: `exp`, `aud`, `scope`
- 사용자 동의 확인: `user_id + client_id + requested_scope`
- 최소 권한: tool마다 필요한 scope 분리
- 감사 로그: 모든 중요 이벤트 기록
- 속도 제한: 간단한 token bucket
- 성능 최적화: 처리 시간(ms) 측정
- 에러 처리: 구조화된 오류 응답 반환


In [1]:
# 1) 설치
!pip -q install -U fastmcp


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.6/728.6 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.3/204.3 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.3/152.3 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 15.0 MB/s eta 0:00:00


In [2]:
# 2) 기본 import
from fastmcp import FastMCP, Client
from pathlib import Path
from datetime import datetime, timedelta, timezone
import sqlite3
import time
import uuid
import json
from collections import defaultdict

print("import 완료")


import 완료


## 1) 샘플 DB 생성

이번 실습에서는 다음 두 테이블을 사용합니다.

- `reports`: 읽기/내보내기 대상 보고서
- `audit_logs`: 감사 로그


In [3]:
DB_PATH = Path("module7_production_hardening.db")

def init_db():
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()

    cur.execute("""
        CREATE TABLE IF NOT EXISTS reports (
            id INTEGER PRIMARY KEY,
            title TEXT NOT NULL,
            summary TEXT NOT NULL,
            body TEXT NOT NULL,
            sensitivity TEXT NOT NULL
        )
    """)

    cur.execute("""
        CREATE TABLE IF NOT EXISTS audit_logs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            ts TEXT NOT NULL,
            correlation_id TEXT NOT NULL,
            user_id TEXT NOT NULL,
            client_id TEXT NOT NULL,
            action TEXT NOT NULL,
            status TEXT NOT NULL,
            details TEXT NOT NULL,
            elapsed_ms REAL NOT NULL
        )
    """)

    cur.execute("DELETE FROM reports")
    cur.execute("DELETE FROM audit_logs")

    sample_reports = [
        (1, "Q1 Sales Summary", "Q1 매출은 전분기 대비 8% 증가했습니다.", "상세 매출 내역과 지역별 분석...", "internal"),
        (2, "Sensitive Ops Report", "운영 리스크와 대응 현황 요약", "민감한 운영 세부 내용...", "confidential"),
    ]
    cur.executemany(
        "INSERT INTO reports (id, title, summary, body, sensitivity) VALUES (?, ?, ?, ?, ?)",
        sample_reports
    )

    conn.commit()
    conn.close()

init_db()
print(f"DB initialized: {DB_PATH.resolve()}")


DB initialized: /content/module7_production_hardening.db


## 2) 보안 및 운영 유틸리티 정의

여기서 다음을 구현합니다.

- **Mock access tokens**
- **Consent registry**
- **Token bucket rate limiter**
- **구조화된 auth/permission 검증**
- **감사 로그 기록 함수**


In [4]:
UTC = timezone.utc

def now_utc():
    return datetime.now(UTC)

def build_mock_tokens():
    future = int((now_utc() + timedelta(hours=1)).timestamp())
    past = int((now_utc() - timedelta(hours=1)).timestamp())

    return {
        "token_read_alice": {
            "sub": "alice",
            "aud": "module7-demo",
            "scope": ["reports:read"],
            "exp": future,
            "client_id": "analytics-ui"
        },
        "token_export_alice": {
            "sub": "alice",
            "aud": "module7-demo",
            "scope": ["reports:read", "reports:export"],
            "exp": future,
            "client_id": "analytics-ui"
        },
        "token_admin_bob": {
            "sub": "bob",
            "aud": "module7-demo",
            "scope": ["reports:read", "reports:export", "audit:read"],
            "exp": future,
            "client_id": "ops-console"
        },
        "token_expired": {
            "sub": "alice",
            "aud": "module7-demo",
            "scope": ["reports:read"],
            "exp": past,
            "client_id": "analytics-ui"
        },
        "token_wrong_audience": {
            "sub": "alice",
            "aud": "other-service",
            "scope": ["reports:read"],
            "exp": future,
            "client_id": "analytics-ui"
        }
    }

TOKENS = build_mock_tokens()

class ConsentRegistry:
    def __init__(self):
        self._grants = defaultdict(set)

    def grant(self, user_id: str, client_id: str, scopes: list[str]):
        self._grants[(user_id, client_id)].update(scopes)

    def has_consent(self, user_id: str, client_id: str, scope: str) -> bool:
        return scope in self._grants[(user_id, client_id)]

    def dump(self):
        return {
            f"{user_id}:{client_id}": sorted(list(scopes))
            for (user_id, client_id), scopes in self._grants.items()
        }

CONSENT = ConsentRegistry()

class TokenBucketRateLimiter:
    def __init__(self, capacity: int, refill_per_sec: float):
        self.capacity = capacity
        self.refill_per_sec = refill_per_sec
        self.state = {}

    def allow(self, key: str, cost: int = 1) -> bool:
        now = time.monotonic()
        tokens, last_ts = self.state.get(key, (self.capacity, now))

        elapsed = now - last_ts
        tokens = min(self.capacity, tokens + elapsed * self.refill_per_sec)

        if tokens < cost:
            self.state[key] = (tokens, now)
            return False

        tokens -= cost
        self.state[key] = (tokens, now)
        return True

RATE_LIMITER = TokenBucketRateLimiter(capacity=3, refill_per_sec=0.5)

def audit_log(correlation_id: str, user_id: str, client_id: str, action: str, status: str, details: dict, elapsed_ms: float):
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute(
        """
        INSERT INTO audit_logs (ts, correlation_id, user_id, client_id, action, status, details, elapsed_ms)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """,
        (
            now_utc().isoformat(),
            correlation_id,
            user_id,
            client_id,
            action,
            status,
            json.dumps(details, ensure_ascii=False),
            elapsed_ms
        )
    )
    conn.commit()
    conn.close()

class AuthError(Exception):
    pass

class PermissionErrorMCP(Exception):
    pass

class RateLimitExceeded(Exception):
    pass

def validate_access_token(access_token: str, required_scopes: list[str], audience: str = "module7-demo") -> dict:
    claims = TOKENS.get(access_token)
    if not claims:
        raise AuthError("Unknown access token")

    if claims["exp"] < int(now_utc().timestamp()):
        raise AuthError("Access token has expired")

    if claims["aud"] != audience:
        raise AuthError(f"Invalid audience: expected {audience}, got {claims['aud']}")

    granted_scopes = set(claims["scope"])
    missing = [s for s in required_scopes if s not in granted_scopes]
    if missing:
        raise PermissionErrorMCP(f"Missing required scopes: {missing}")

    return claims

def result_ok(action: str, correlation_id: str, user_id: str, client_id: str, payload: dict, elapsed_ms: float):
    audit_log(correlation_id, user_id, client_id, action, "ok", payload, elapsed_ms)
    return {
        "ok": True,
        "action": action,
        "correlation_id": correlation_id,
        "elapsed_ms": round(elapsed_ms, 2),
        **payload
    }

def result_error(action: str, correlation_id: str, user_id: str, client_id: str, code: str, message: str, elapsed_ms: float, extra: dict | None = None):
    payload = {
        "ok": False,
        "action": action,
        "error": {
            "code": code,
            "message": message
        }
    }
    if extra:
        payload["error"]["extra"] = extra

    audit_log(correlation_id, user_id, client_id, action, "error", payload, elapsed_ms)
    payload["correlation_id"] = correlation_id
    payload["elapsed_ms"] = round(elapsed_ms, 2)
    return payload

print("보안/운영 유틸리티 준비 완료")


보안/운영 유틸리티 준비 완료


## 3) MCP Server 정의

이번 실습 서버는 다음 tool을 제공합니다.

1. `get_report_summary`
   - scope: `reports:read`
   - read 전용 access

2. `export_sensitive_report`
   - scope: `reports:export`
   - **추가로 사용자 동의** 필요

3. `list_audit_logs`
   - scope: `audit:read`
   - 운영/감사 관리자 전용


In [5]:
mcp = FastMCP("Module7-Hardening-Demo")

@mcp.tool
def get_report_summary(report_id: int, access_token: str) -> dict:
    """보고서 summary를 읽습니다."""
    action = "get_report_summary"
    correlation_id = str(uuid.uuid4())
    started = time.perf_counter()

    user_id = "unknown"
    client_id = "unknown"

    try:
        claims = validate_access_token(access_token, ["reports:read"])
        user_id = claims["sub"]
        client_id = claims["client_id"]

        rate_key = f"{user_id}:{action}"
        if not RATE_LIMITER.allow(rate_key):
            raise RateLimitExceeded("Rate limit exceeded for get_report_summary")

        conn = sqlite3.connect(DB_PATH)
        cur = conn.cursor()
        cur.execute("SELECT id, title, summary, sensitivity FROM reports WHERE id = ?", (report_id,))
        row = cur.fetchone()
        conn.close()

        if row is None:
            raise ValueError(f"Report id={report_id} not found")

        payload = {
            "report": {
                "id": row[0],
                "title": row[1],
                "summary": row[2],
                "sensitivity": row[3]
            }
        }
        elapsed_ms = (time.perf_counter() - started) * 1000
        return result_ok(action, correlation_id, user_id, client_id, payload, elapsed_ms)

    except AuthError as e:
        elapsed_ms = (time.perf_counter() - started) * 1000
        return result_error(action, correlation_id, user_id, client_id, "AUTH_ERROR", str(e), elapsed_ms)
    except PermissionErrorMCP as e:
        elapsed_ms = (time.perf_counter() - started) * 1000
        return result_error(action, correlation_id, user_id, client_id, "PERMISSION_DENIED", str(e), elapsed_ms)
    except RateLimitExceeded as e:
        elapsed_ms = (time.perf_counter() - started) * 1000
        return result_error(action, correlation_id, user_id, client_id, "RATE_LIMIT", str(e), elapsed_ms)
    except Exception as e:
        elapsed_ms = (time.perf_counter() - started) * 1000
        return result_error(action, correlation_id, user_id, client_id, "INTERNAL_ERROR", str(e), elapsed_ms)

@mcp.tool
def export_sensitive_report(report_id: int, access_token: str) -> dict:
    """민감 보고서를 export합니다. 사용자 동의와 export scope가 모두 필요합니다."""
    action = "export_sensitive_report"
    correlation_id = str(uuid.uuid4())
    started = time.perf_counter()

    user_id = "unknown"
    client_id = "unknown"

    try:
        claims = validate_access_token(access_token, ["reports:export"])
        user_id = claims["sub"]
        client_id = claims["client_id"]

        if not CONSENT.has_consent(user_id, client_id, "reports:export"):
            raise PermissionErrorMCP(
                f"Explicit user consent is required for scope 'reports:export' (user={user_id}, client={client_id})"
            )

        rate_key = f"{user_id}:{action}"
        if not RATE_LIMITER.allow(rate_key):
            raise RateLimitExceeded("Rate limit exceeded for export_sensitive_report")

        conn = sqlite3.connect(DB_PATH)
        cur = conn.cursor()
        cur.execute("SELECT id, title, body, sensitivity FROM reports WHERE id = ?", (report_id,))
        row = cur.fetchone()
        conn.close()

        if row is None:
            raise ValueError(f"Report id={report_id} not found")

        if row[3] != "confidential":
            raise ValueError("Only confidential reports are exportable in this demo")

        payload = {
            "exported_report": {
                "id": row[0],
                "title": row[1],
                "body_preview": row[2][:80],
                "sensitivity": row[3]
            },
            "consent_checked": True
        }
        elapsed_ms = (time.perf_counter() - started) * 1000
        return result_ok(action, correlation_id, user_id, client_id, payload, elapsed_ms)

    except AuthError as e:
        elapsed_ms = (time.perf_counter() - started) * 1000
        return result_error(action, correlation_id, user_id, client_id, "AUTH_ERROR", str(e), elapsed_ms)
    except PermissionErrorMCP as e:
        elapsed_ms = (time.perf_counter() - started) * 1000
        return result_error(action, correlation_id, user_id, client_id, "PERMISSION_DENIED", str(e), elapsed_ms)
    except RateLimitExceeded as e:
        elapsed_ms = (time.perf_counter() - started) * 1000
        return result_error(action, correlation_id, user_id, client_id, "RATE_LIMIT", str(e), elapsed_ms)
    except Exception as e:
        elapsed_ms = (time.perf_counter() - started) * 1000
        return result_error(action, correlation_id, user_id, client_id, "INTERNAL_ERROR", str(e), elapsed_ms)

@mcp.tool
def list_audit_logs(limit: int, access_token: str) -> dict:
    """감사 로그를 조회합니다. 관리자 scope가 필요합니다."""
    action = "list_audit_logs"
    correlation_id = str(uuid.uuid4())
    started = time.perf_counter()

    user_id = "unknown"
    client_id = "unknown"

    try:
        claims = validate_access_token(access_token, ["audit:read"])
        user_id = claims["sub"]
        client_id = claims["client_id"]

        rate_key = f"{user_id}:{action}"
        if not RATE_LIMITER.allow(rate_key):
            raise RateLimitExceeded("Rate limit exceeded for list_audit_logs")

        conn = sqlite3.connect(DB_PATH)
        cur = conn.cursor()
        cur.execute(
            """
            SELECT ts, correlation_id, user_id, client_id, action, status, details, elapsed_ms
            FROM audit_logs
            ORDER BY id DESC
            LIMIT ?
            """,
            (limit,)
        )
        rows = cur.fetchall()
        conn.close()

        payload = {
            "logs": [
                {
                    "ts": row[0],
                    "correlation_id": row[1],
                    "user_id": row[2],
                    "client_id": row[3],
                    "action": row[4],
                    "status": row[5],
                    "details": json.loads(row[6]),
                    "elapsed_ms": row[7]
                }
                for row in rows
            ]
        }
        elapsed_ms = (time.perf_counter() - started) * 1000
        return result_ok(action, correlation_id, user_id, client_id, payload, elapsed_ms)

    except AuthError as e:
        elapsed_ms = (time.perf_counter() - started) * 1000
        return result_error(action, correlation_id, user_id, client_id, "AUTH_ERROR", str(e), elapsed_ms)
    except PermissionErrorMCP as e:
        elapsed_ms = (time.perf_counter() - started) * 1000
        return result_error(action, correlation_id, user_id, client_id, "PERMISSION_DENIED", str(e), elapsed_ms)
    except RateLimitExceeded as e:
        elapsed_ms = (time.perf_counter() - started) * 1000
        return result_error(action, correlation_id, user_id, client_id, "RATE_LIMIT", str(e), elapsed_ms)
    except Exception as e:
        elapsed_ms = (time.perf_counter() - started) * 1000
        return result_error(action, correlation_id, user_id, client_id, "INTERNAL_ERROR", str(e), elapsed_ms)

print("MCP server 정의 완료")


MCP server 정의 완료


## 4) Client 생성


In [6]:
client = Client(mcp)
print("Client 준비 완료")


Client 준비 완료


## 5) 도구 목록 확인


In [7]:
async def show_tools():
    async with client:
        tools = await client.list_tools()
        print(f"도구 개수: {len(tools)}")
        for t in tools:
            print({
                "name": getattr(t, "name", None),
                "description": getattr(t, "description", None)
            })

await show_tools()


도구 개수: 3
{'name': 'get_report_summary', 'description': '보고서 summary를 읽습니다.'}
{'name': 'export_sensitive_report', 'description': '민감 보고서를 export합니다. 사용자 동의와 export scope가 모두 필요합니다.'}
{'name': 'list_audit_logs', 'description': '감사 로그를 조회합니다. 관리자 scope가 필요합니다.'}


## 6) 정상 read 호출 테스트

`token_read_alice` 는 `reports:read` scope만 있습니다.  
따라서 **summary 읽기**는 가능하지만, export나 audit 조회는 할 수 없습니다.


In [8]:
async def demo_normal_read():
    async with client:
        result = await client.call_tool(
            "get_report_summary",
            {"report_id": 1, "access_token": "token_read_alice"}
        )
        print("data:", getattr(result, "data", None))
        print("content:", getattr(result, "content", None))

await demo_normal_read()


data: {'ok': True, 'action': 'get_report_summary', 'correlation_id': 'eb7cff51-ed9b-4401-9dc7-c72ae728c5f2', 'elapsed_ms': 0.63, 'report': {'id': 1, 'title': 'Q1 Sales Summary', 'summary': 'Q1 매출은 전분기 대비 8% 증가했습니다.', 'sensitivity': 'internal'}}
content: [TextContent(type='text', text='{"ok":true,"action":"get_report_summary","correlation_id":"eb7cff51-ed9b-4401-9dc7-c72ae728c5f2","elapsed_ms":0.63,"report":{"id":1,"title":"Q1 Sales Summary","summary":"Q1 매출은 전분기 대비 8% 증가했습니다.","sensitivity":"internal"}}', annotations=None, meta=None)]


## 7) 동의 없이 export 호출 → 실패 확인

`token_export_alice` 는 export scope를 가지고 있어도,  
이 demo에서는 **민감 작업은 별도 사용자 동의**가 있어야 합니다.


In [9]:
async def demo_export_without_consent():
    async with client:
        result = await client.call_tool(
            "export_sensitive_report",
            {"report_id": 2, "access_token": "token_export_alice"}
        )
        print("data:", getattr(result, "data", None))
        print("content:", getattr(result, "content", None))

await demo_export_without_consent()


data: {'ok': False, 'action': 'export_sensitive_report', 'error': {'code': 'PERMISSION_DENIED', 'message': "Explicit user consent is required for scope 'reports:export' (user=alice, client=analytics-ui)"}, 'correlation_id': '346aa14a-884e-4a13-8930-ede5ac3bb72f', 'elapsed_ms': 0.04}
content: [TextContent(type='text', text='{"ok":false,"action":"export_sensitive_report","error":{"code":"PERMISSION_DENIED","message":"Explicit user consent is required for scope \'reports:export\' (user=alice, client=analytics-ui)"},"correlation_id":"346aa14a-884e-4a13-8930-ede5ac3bb72f","elapsed_ms":0.04}', annotations=None, meta=None)]


## 8) 사용자 동의 부여

여기서는 실제 UI 대신, notebook 셀에서 동의를 기록합니다.


In [10]:
CONSENT.grant(user_id="alice", client_id="analytics-ui", scopes=["reports:export"])
print("현재 consent registry:")
print(json.dumps(CONSENT.dump(), ensure_ascii=False, indent=2))


현재 consent registry:
{
  "alice:analytics-ui": [
    "reports:export"
  ]
}


## 9) 동의 후 export 호출 → 성공 확인


In [11]:
async def demo_export_with_consent():
    async with client:
        result = await client.call_tool(
            "export_sensitive_report",
            {"report_id": 2, "access_token": "token_export_alice"}
        )
        print("data:", getattr(result, "data", None))
        print("content:", getattr(result, "content", None))

await demo_export_with_consent()


data: {'ok': True, 'action': 'export_sensitive_report', 'correlation_id': 'bf2243d0-9d38-40f8-8f8a-cf8c5046c483', 'elapsed_ms': 0.67, 'exported_report': {'id': 2, 'title': 'Sensitive Ops Report', 'body_preview': '민감한 운영 세부 내용...', 'sensitivity': 'confidential'}, 'consent_checked': True}
content: [TextContent(type='text', text='{"ok":true,"action":"export_sensitive_report","correlation_id":"bf2243d0-9d38-40f8-8f8a-cf8c5046c483","elapsed_ms":0.67,"exported_report":{"id":2,"title":"Sensitive Ops Report","body_preview":"민감한 운영 세부 내용...","sensitivity":"confidential"},"consent_checked":true}', annotations=None, meta=None)]


## 10) 만료 토큰 / audience 오류 / scope 부족 테스트


In [12]:
async def demo_auth_failures():
    async with client:
        print("=== expired token ===")
        r1 = await client.call_tool(
            "get_report_summary",
            {"report_id": 1, "access_token": "token_expired"}
        )
        print(getattr(r1, "data", None))

        print("\n=== wrong audience ===")
        r2 = await client.call_tool(
            "get_report_summary",
            {"report_id": 1, "access_token": "token_wrong_audience"}
        )
        print(getattr(r2, "data", None))

        print("\n=== insufficient scope for audit ===")
        r3 = await client.call_tool(
            "list_audit_logs",
            {"limit": 5, "access_token": "token_read_alice"}
        )
        print(getattr(r3, "data", None))

await demo_auth_failures()


=== expired token ===
{'ok': False, 'action': 'get_report_summary', 'error': {'code': 'AUTH_ERROR', 'message': 'Access token has expired'}, 'correlation_id': '6ab9c608-b2ee-4f60-b2a4-6a827768355e', 'elapsed_ms': 0.03}

=== wrong audience ===
{'ok': False, 'action': 'get_report_summary', 'error': {'code': 'AUTH_ERROR', 'message': 'Invalid audience: expected module7-demo, got other-service'}, 'correlation_id': '04f00207-83c9-4071-9a7c-8c6910f33787', 'elapsed_ms': 0.03}

=== insufficient scope for audit ===
{'ok': False, 'action': 'list_audit_logs', 'error': {'code': 'PERMISSION_DENIED', 'message': "Missing required scopes: ['audit:read']"}, 'correlation_id': 'f38010f5-6910-4875-b198-9ddaddc4da46', 'elapsed_ms': 0.04}


## 11) Rate Limit 초과 테스트

`get_report_summary` 는 같은 사용자/동작 기준으로 빠르게 반복 호출하면  
간단한 token bucket 제한에 걸리도록 구성했습니다.


In [13]:
async def demo_rate_limit():
    async with client:
        for i in range(5):
            result = await client.call_tool(
                "get_report_summary",
                {"report_id": 1, "access_token": "token_read_alice"}
            )
            print(f"call #{i+1}")
            print(getattr(result, "data", None))
            print("-" * 60)

await demo_rate_limit()


call #1
{'ok': True, 'action': 'get_report_summary', 'correlation_id': 'd907a9af-3974-4e90-add7-10846a2ce779', 'elapsed_ms': 0.73, 'report': {'id': 1, 'title': 'Q1 Sales Summary', 'summary': 'Q1 매출은 전분기 대비 8% 증가했습니다.', 'sensitivity': 'internal'}}
------------------------------------------------------------
call #2
{'ok': True, 'action': 'get_report_summary', 'correlation_id': '8664efff-bdb4-4dd2-aa4a-9174f46a989d', 'elapsed_ms': 0.67, 'report': {'id': 1, 'title': 'Q1 Sales Summary', 'summary': 'Q1 매출은 전분기 대비 8% 증가했습니다.', 'sensitivity': 'internal'}}
------------------------------------------------------------
call #3
{'ok': False, 'action': 'get_report_summary', 'error': {'code': 'RATE_LIMIT', 'message': 'Rate limit exceeded for get_report_summary'}, 'correlation_id': 'c737b13a-2a90-4d2d-ab7f-f93ac9391f2d', 'elapsed_ms': 0.04}
------------------------------------------------------------
call #4
{'ok': False, 'action': 'get_report_summary', 'error': {'code': 'RATE_LIMIT', 'message': 'Rat

## 12) Audit Log 조회

이 기능은 `audit:read` scope가 필요하므로,  
관리자 토큰인 `token_admin_bob` 로만 성공합니다.


In [14]:
async def demo_audit_read():
    async with client:
        result = await client.call_tool(
            "list_audit_logs",
            {"limit": 10, "access_token": "token_admin_bob"}
        )
        print(json.dumps(getattr(result, "data", None), ensure_ascii=False, indent=2))

await demo_audit_read()


{
  "ok": true,
  "action": "list_audit_logs",
  "correlation_id": "5c198fc1-4b76-4a5b-8bac-af4db18b116b",
  "elapsed_ms": 2.51,
  "logs": [
    {
      "ts": "2026-04-15T16:50:21.056266+00:00",
      "correlation_id": "4c1e9adb-28d0-47f8-863a-c7c4c66a0cfb",
      "user_id": "alice",
      "client_id": "analytics-ui",
      "action": "get_report_summary",
      "status": "error",
      "details": {
        "ok": false,
        "action": "get_report_summary",
        "error": {
          "code": "RATE_LIMIT",
          "message": "Rate limit exceeded for get_report_summary"
        }
      },
      "elapsed_ms": 0.03993500001797656
    },
    {
      "ts": "2026-04-15T16:50:21.039505+00:00",
      "correlation_id": "fe02b8f8-c497-43c3-857f-a04ed144bd20",
      "user_id": "alice",
      "client_id": "analytics-ui",
      "action": "get_report_summary",
      "status": "error",
      "details": {
        "ok": false,
        "action": "get_report_summary",
        "error": {
          "co

## 13) DB에서 감사 로그 직접 확인


In [15]:
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
cur.execute("""
    SELECT id, ts, user_id, client_id, action, status, elapsed_ms
    FROM audit_logs
    ORDER BY id DESC
    LIMIT 10
""")
rows = cur.fetchall()
conn.close()

print("최근 감사 로그:")
for row in rows:
    print(row)


최근 감사 로그:
(12, '2026-04-15T16:50:21.099431+00:00', 'bob', 'ops-console', 'list_audit_logs', 'ok', 2.5064390000579806)
(11, '2026-04-15T16:50:21.056266+00:00', 'alice', 'analytics-ui', 'get_report_summary', 'error', 0.03993500001797656)
(10, '2026-04-15T16:50:21.039505+00:00', 'alice', 'analytics-ui', 'get_report_summary', 'error', 0.042519999965406896)
(9, '2026-04-15T16:50:21.021009+00:00', 'alice', 'analytics-ui', 'get_report_summary', 'error', 0.03697499994359532)
(8, '2026-04-15T16:50:21.001337+00:00', 'alice', 'analytics-ui', 'get_report_summary', 'ok', 0.6725120000510287)
(7, '2026-04-15T16:50:20.976188+00:00', 'alice', 'analytics-ui', 'get_report_summary', 'ok', 0.7255189999568756)
(6, '2026-04-15T16:50:20.931476+00:00', 'unknown', 'unknown', 'list_audit_logs', 'error', 0.03887800005486497)
(5, '2026-04-15T16:50:20.913789+00:00', 'unknown', 'unknown', 'get_report_summary', 'error', 0.02720899999530957)
(4, '2026-04-15T16:50:20.884417+00:00', 'unknown', 'unknown', 'get_report_sum

## 14) 실습 정리

이번 실습에서 확인한 핵심:

1. **최소 권한(Least Privilege)**: tool마다 필요한 scope를 분리해야 한다.
2. **사용자 동의(Consent)**: 민감한 작업은 scope만으로 끝내지 않고 별도 동의를 요구할 수 있다.
3. **OAuth 2.1 스타일 검증 핵심**: `exp`, `aud`, `scope`를 확인해야 한다.
4. **운영 안정화**: 에러를 구조화하고, 감사 로그와 correlation_id를 남겨야 한다.
5. **Rate Limiting**: 서버 남용과 급격한 부하를 막는 기본 방어선이다.
6. **성능 측정**: 각 요청의 elapsed time을 기록해 병목을 찾을 수 있다.

추가 확장 아이디어:
- HTTP transport + 실제 OAuth provider 연동
- FastMCP auth provider 사용
- 로그를 파일/ELK/Cloud Logging으로 전송
- Redis 기반 분산 rate limit 적용
